## First look playbook

Do not design the warehouse yet. Run the load cells, then work top to bottom through **First look: grain, fact, dim** at the bottom of this notebook.

On any new dataset, always ask the same three questions:
1. One row = what? (**grain**)
2. Did something *happen*? (**fact**)
3. Is this a *thing we describe*? (**dimension**)

Start Explore Data

In [5]:
import pandas as pd

ROOT_DIR = 'D:/Coding/Instacart Market Basket Data Warehouse/datasets/'

Data

In [6]:
aisles = pd.read_csv(ROOT_DIR + 'aisles.csv')
departments = pd.read_csv(ROOT_DIR + 'departments.csv')
orders = pd.read_csv(ROOT_DIR + 'orders.csv')
order_products_prior = pd.read_csv(ROOT_DIR + 'order_products__prior.csv')
order_products_train = pd.read_csv(ROOT_DIR + 'order_products__train.csv')
products = pd.read_csv(ROOT_DIR + 'products.csv')

Checking above dataframes

In [7]:
aisles.head()

,aisle_id,aisle
0,1,prepared soups salads
1,2,specialty cheeses
2,3,energy granola bars
3,4,instant foods
4,5,marinades meat preparation


In [8]:
departments.head()

,department_id,department
0,1,frozen
1,2,other
2,3,bakery
3,4,produce
4,5,alcohol


In [9]:
order_products_prior.head()

,order_id,product_id,add_to_cart_order,reordered
0,2,33120,1,1
1,2,28985,2,1
2,2,9327,3,0
3,2,45918,4,1
4,2,30035,5,0


In [10]:
order_products_train.head()

,order_id,product_id,add_to_cart_order,reordered
0,1,49302,1,1
1,1,11109,2,1
2,1,10246,3,0
3,1,49683,4,0
4,1,43633,5,1


In [11]:
orders.head()

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,8,NaN
1,2398795,1,prior,2,3,7,15.0
2,473747,1,prior,3,3,12,21.0
3,2254736,1,prior,4,4,7,29.0
4,431534,1,prior,5,4,15,28.0


In [12]:
products.head()

,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13


# First look: grain, fact, dim

When you first open a dataset, do **not** draw a warehouse yet.

Do the same 5 steps on every table:
1. List tables and columns
2. Say the **grain** in one sentence: *one row is one _____*
3. Find which column(s) make a row unique
4. Decide: is this a **thing** (dim) or an **event** (fact)?
5. Only then choose which IDs belong on the fact

Run the CSV load cells above first.

In [13]:
# Step 1 — list every table: rows, columns, first rows
tables = {
    "aisles": aisles,
    "departments": departments,
    "products": products,
    "orders": orders,
    "order_products_prior": order_products_prior,
    "order_products_train": order_products_train,
}

for name, df in tables.items():
    print("=" * 72)
    print(name, "| rows:", f"{len(df):,}", "| columns:", list(df.columns))
    print(df.head(2).to_string(index=False))
    print()

aisles | rows: 134 | columns: ['aisle_id', 'aisle']
 aisle_id                 aisle
        1 prepared soups salads
        2     specialty cheeses

departments | rows: 21 | columns: ['department_id', 'department']
 department_id department
             1     frozen
             2      other

products | rows: 49,688 | columns: ['product_id', 'product_name', 'aisle_id', 'department_id']
 product_id               product_name  aisle_id  department_id
          1 Chocolate Sandwich Cookies        61             19
          2           All-Seasons Salt       104             13

orders | rows: 3,421,083 | columns: ['order_id', 'user_id', 'eval_set', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order']
 order_id  user_id eval_set  order_number  order_dow  order_hour_of_day  days_since_prior_order
  2539329        1    prior             1          2                  8                     NaN
  2398795        1    prior             2          3                  7       

## Step 2 — grain: finish this sentence

**One row in this table is one _____.**

How to find it:
- If an `id` is unique, one row is usually that thing (`product_id` unique → one product).
- If the same `id` repeats, grain is smaller (`order_id` repeats → not one order).

Run the next cell. Watch `order_id` on `order_products_prior`: it is **not** unique.

In [14]:
def is_unique(df, cols):
    cols = cols if isinstance(cols, list) else [cols]
    return not df.duplicated(cols).any()

checks = [
    ("aisles", aisles, "aisle_id"),
    ("departments", departments, "department_id"),
    ("products", products, "product_id"),
    ("orders", orders, "order_id"),
    ("orders user_id", orders, "user_id"),
    ("order_products_prior order_id", order_products_prior, "order_id"),
    ("order_products_prior product_id", order_products_prior, "product_id"),
    ("order_products_prior order+product", order_products_prior, ["order_id", "product_id"]),
]

for label, df, key in checks:
    print(f"{label:42} unique? {is_unique(df, key)}")

aisles                                     unique? True
departments                                unique? True
products                                   unique? True
orders                                     unique? True
orders user_id                             unique? False
order_products_prior order_id              unique? False
order_products_prior product_id            unique? False
order_products_prior order+product         unique? True


Expected reading of that output:

| Table | Unique key | Grain you should write down |
|---|---|---|
| aisles | `aisle_id` | one aisle |
| departments | `department_id` | one department |
| products | `product_id` | one product |
| orders | `order_id` | one order (the basket **header**) |
| order_products_prior | `order_id` + `product_id` | one **product inside one order** |

`user_id` on orders is not unique because one user places many orders.

## Step 3 — event or thing?

Ask of each table: **can I count this row as something that happened?**

- **Thing (dimension):** a person, product, aisle, day. You describe it. Names, categories.
- **Event (fact):** a purchase line, a click, a payment. You count it.

On Instacart:
- `products`, `aisles`, `departments` → things → **dim**
- `orders` → looks like an event, but it is only the **header** of a basket
- `order_products_*` → the real event: this product was in this order → **fact**

Trap: do not pick `orders` as the fact. One order contains many products. Market-basket questions need the line items.

## Step 4 — prove grain with one example

Pick `order_id == 2`. If `orders` has 1 row and `order_products_prior` has many rows, the fact grain is the many rows.

In [ ]:
EXAMPLE_ORDER = 2

print("HEADER — orders (should be 1 row)")
display(orders.query("order_id == @EXAMPLE_ORDER"))

print("LINES — order_products_prior (many rows = fact grain)")
lines = order_products_prior.query("order_id == @EXAMPLE_ORDER")
display(lines)

print("NAMES — join products so you can read the lines (names are dim, not fact)")
display(
    lines.merge(products, on="product_id")[
        ["order_id", "product_id", "product_name", "add_to_cart_order", "reordered"]
    ]
)

## Step 5 — which IDs go on the fact?

The fact **copies IDs**. It does not invent them. It does not store names.

1. Start from the event file: `order_id`, `product_id`, `reordered`, `add_to_cart_order`
2. Join `orders` on `order_id` to copy `user_id`, `order_dow`, `order_hour_of_day`, `order_number`, `days_since_prior_order`, `eval_set`
3. Leave `product_name`, `aisle_id`, `department_id` on **dim_product** (join later when you query)

That is the whole method:
**look → grain → unique key → event vs thing → copy event IDs into the fact.**